# UC10 — Asistente Inteligente de Consultas de Póliza

Asistente Q&A basado en RAG sobre documentación de pólizas usando Cortex Search.
**Valor de negocio**: Respuestas precisas en <3s para agentes del contact center.

---

## Paso 1: Configuración del Entorno

In [ ]:
CREATE DATABASE IF NOT EXISTS ASISTENTE_POLIZAS_DB;
USE SCHEMA ASISTENTE_POLIZAS_DB.PUBLIC;
CREATE WAREHOUSE IF NOT EXISTS COMPUTE_WH WITH WAREHOUSE_SIZE='SMALL' AUTO_SUSPEND=60 AUTO_RESUME=TRUE;
USE WAREHOUSE COMPUTE_WH;

---

## Paso 2: Generar Base de Conocimiento de Pólizas

In [ ]:
CREATE OR REPLACE TABLE documentos_poliza AS
SELECT column1 AS doc_id, column2 AS producto, column3 AS seccion, column4 AS titulo, column5 AS contenido FROM VALUES
('D001','Auto','Coberturas','Responsabilidad Civil','La póliza de auto cubre la responsabilidad civil obligatoria hasta 70 millones de euros por daños corporales y 15 millones por daños materiales. Incluye defensa jurídica y fianzas.'),
('D002','Auto','Coberturas','Robo','La cobertura de robo incluye el hurto total del vehículo, robo de accesorios fijos y daños causados durante el intento de robo. Franquicia aplicable: 150€.'),
('D003','Auto','Coberturas','Lunas','La cobertura de lunas incluye parabrisas, luneta trasera y ventanillas laterales. Reparación sin franquicia. Sustitución con franquicia de 100€.'),
('D004','Auto','Plazos','Notificación Siniestro','El asegurado debe notificar el siniestro en un plazo máximo de 7 días hábiles desde su ocurrencia. En caso de robo, debe presentar denuncia policial en 24 horas.'),
('D005','Auto','Asistencia','Asistencia en Carretera','Asistencia 24h en toda España y Europa. Incluye grúa hasta 50km, vehículo de sustitución por 5 días, alojamiento de emergencia y continuación de viaje.'),
('D006','Hogar','Coberturas','Incendio','Cubre daños por incendio, explosión, rayo y caída de aeronaves. Incluye continente y contenido. Valor de reconstrucción a nuevo. Sin franquicia.'),
('D007','Hogar','Coberturas','Agua','Daños por agua: rotura de tuberías, desbordamiento, filtraciones. Incluye localización de avería, reparación y daños estéticos. Franquicia: 100€.'),
('D008','Hogar','Coberturas','Robo en Vivienda','Cobertura de robo y expoliación en domicilio. Límite por objeto de valor: 3.000€. Límite total contenido: según capital asegurado. Requiere medidas de seguridad mínimas.'),
('D009','Hogar','Plazos','Comunicación Daños','Los daños deben comunicarse en un máximo de 5 días hábiles. En caso de robo, presentar denuncia policial inmediata y notificar a la aseguradora en 24 horas.'),
('D010','Hogar','Coberturas','Responsabilidad Civil','RC del hogar cubre daños a terceros causados por el inmueble o sus ocupantes. Límite: 300.000€. Incluye daños por agua a vecinos, caída de objetos y mascotas.'),
('D011','Vida','Coberturas','Fallecimiento','Capital asegurado pagadero a beneficiarios en caso de fallecimiento del asegurado por cualquier causa. Carencia: 12 meses para suicidio. Sin carencia para accidente.'),
('D012','Vida','Coberturas','Invalidez','Invalidez permanente absoluta: pago del 100% del capital. Invalidez parcial: según baremo. Incluye anticipo por enfermedad grave.'),
('D013','Salud','Coberturas','Hospitalización','Cobertura de hospitalización médica y quirúrgica sin límite de estancia. Habitación individual. Incluye UCI, quirófano y honorarios médicos.'),
('D014','Salud','Coberturas','Consultas Especialistas','Acceso a cuadro médico de especialistas sin lista de espera. Copago de 10€ por consulta. Incluye pruebas diagnósticas prescritas.'),
('D015','Salud','Plazos','Carencias','Carencias: 6 meses para hospitalización, 8 meses para parto, 12 meses para prótesis dentales. Sin carencia para urgencias y accidentes.'),
('D016','Auto','Exclusiones','Exclusiones Generales','No se cubren: conducción bajo efectos de alcohol o drogas, uso del vehículo para competición, daños intencionados, vehículos sin ITV vigente.'),
('D017','Hogar','Exclusiones','Exclusiones Hogar','No se cubren: daños por falta de mantenimiento, humedades por condensación, plagas de insectos, desgaste natural, actos de guerra.'),
('D018','Auto','FAQ','Cambio de Vehículo','Para cambiar el vehículo asegurado, contacte con su agente o llame al 900 123 456. El cambio se aplica desde la fecha solicitada sin coste adicional si la prima es equivalente.'),
('D019','Hogar','FAQ','Daños por Mascota','Los daños causados por mascotas a terceros están cubiertos por la RC del hogar. Los daños que la mascota cause dentro de la propia vivienda NO están cubiertos.'),
('D020','Salud','FAQ','Segunda Opinión','El seguro de salud incluye servicio de segunda opinión médica para diagnósticos graves. Sin coste adicional. Consulta con especialistas de referencia.')
;

SELECT * FROM documentos_poliza LIMIT 5;

---

## Paso 3: Crear Chunks de Texto para Indexación

In [ ]:
CREATE OR REPLACE TABLE chunks_poliza AS
SELECT
    doc_id || '_C1' AS chunk_id,
    doc_id, producto, seccion,
    contenido AS chunk_texto,
    1 AS chunk_orden
FROM documentos_poliza;

SELECT COUNT(*) AS total_chunks FROM chunks_poliza;

---

## Paso 4: Crear Cortex Search Service

Indexamos los chunks para búsqueda semántica.

In [ ]:
CREATE OR REPLACE CORTEX SEARCH SERVICE buscador_polizas
    ON chunk_texto
    ATTRIBUTES producto, seccion
    WAREHOUSE = COMPUTE_WH
    TARGET_LAG = '1 hour'
    AS (SELECT chunk_texto, producto, seccion, chunk_id, doc_id FROM chunks_poliza);

---

## Paso 5: Implementar Pipeline RAG

Función que busca contexto relevante y genera respuesta con Cortex.

In [ ]:
CREATE OR REPLACE FUNCTION responder_consulta(pregunta VARCHAR)
RETURNS VARCHAR
LANGUAGE SQL
AS $$
    SELECT SNOWFLAKE.CORTEX.COMPLETE('mistral-large2',
        'Eres un asistente experto en seguros. Responde la pregunta del agente basándote SOLO en el contexto proporcionado. Si no encuentras la respuesta, di "No tengo información suficiente". Cita el documento fuente.\n\nContexto:\n' ||
        (SELECT LISTAGG(chunk_texto, '\n---\n') FROM chunks_poliza WHERE LOWER(chunk_texto) LIKE '%' || LOWER(SPLIT_PART(pregunta,' ',1)) || '%' LIMIT 3) ||
        '\n\nPregunta: ' || pregunta || '\n\nRespuesta:'
    )
$$;

SELECT responder_consulta('¿Qué cubre el seguro de hogar en caso de robo?') AS respuesta;

---

## Paso 6: Probar con Preguntas Tipo

In [ ]:
SELECT 'Lunas' AS tema, responder_consulta('¿La rotura de cristales del coche tiene franquicia?') AS respuesta
UNION ALL SELECT 'Plazo', responder_consulta('¿Cuál es el plazo para notificar un siniestro de auto?')
UNION ALL SELECT 'Mascota', responder_consulta('¿Están cubiertos los daños por mascota en el hogar?')
UNION ALL SELECT 'Asistencia', responder_consulta('¿Cómo solicito asistencia en carretera?')
UNION ALL SELECT 'Carencias', responder_consulta('¿Cuánto tiempo de carencia tiene la hospitalización en salud?');

---

## Paso 7: Dashboard del Asistente

In [ ]:

session = get_active_session()
st.title('Asistente de Consultas de Póliza')
st.markdown('### Respuestas IA para Agentes — Cortex Search + RAG')

producto = st.selectbox('Producto', ['Todos','Auto','Hogar','Vida','Salud'])
pregunta = st.text_input('Pregunta del agente:', placeholder='Ej: ¿Qué cubre la póliza de hogar en caso de incendio?')

if pregunta:
    with st.spinner('Buscando respuesta...'):
        result = session.sql(f"SELECT responder_consulta('{pregunta}') AS resp").collect()
        st.markdown('### Respuesta')
        st.write(result[0]['RESP'])

st.markdown('---')
st.subheader('Base de Conocimiento')
docs = session.table('documentos_poliza').to_pandas()
if producto != 'Todos': docs = docs[docs['PRODUCTO']==producto]
st.dataframe(docs[['DOC_ID','PRODUCTO','SECCION','TITULO']], use_container_width=True, height=300)
st.caption('Desarrollado con Snowflake Cortex Search + Streamlit')

---

## Paso 8: Limpieza (Opcional)

In [ ]:
-- DROP DATABASE IF EXISTS ASISTENTE_POLIZAS_DB;
-- DROP WAREHOUSE IF EXISTS COMPUTE_WH;